# CS 1066 Classnotes (20261109)

### As You Get Settled

1.  Start your CS1066 codespace. Go to `https://github.com/<your_username>/cs1066`, and either launch a new or restart the existing codespace for this repo.

2.  In this codespace, run `python3 grab1066.py m15a`. If you get an error, run `cd /workspaces/cs1066` in the terminal and then rerun the grab command.

3.  Use the Qwickly Attendance app to scan the QR code shown in class. Then, consider putting your cellphone away.

----
----

### Today's topics:

1.  Revisit Sid's guest lecture slides to better understand them.
2.  What is a "reasoning" model?
3.  How do "thinking" and tool use interact?
4.  What is a turn in both Anthropic's and Sid's worlds?
5.  Seeing an agentic loop in action
    - Problem-to-be-solved: workflow from messages to calendar events

----
----

### Key Point of Sid's Slides

> Open Sid Pardeshi's talk slides, which you can find [here](https://canvas.harvard.edu/courses/173874/files).

Slide 3 defines an agent's "autonomous work" as "an AI model working in a loop (read, use a tool, check, repeat)."

-   Each loop iteration is defined as _one turn_.
-   An agent requires more turns (i.e., longer runs) to understand / manipulate larger code bases (in Blitzy's domain).
-   Equivalently, as the complexity of your problem grows, an agent will require more turns and longer runs to solve it.

> "Turn" is not a clearly defined or consistently used term, as we'll learn today.

----

### What defines the work done autonomously?

The prompt, of course. 

But as we've seen with our own prompts, not everything an agent does is in our prompt. This is especially true if you're working with a **"reasoning" model**, which is the evolution of "chain-of-thought" approaches.

> A reasoning model has the capacity to "think things through" before giving an answer.

E.g., in creating or modifying a program, the Copilot agent does the following:

-   Reads the program (if it exists) and the other files in the project.
-   Makes a plan for the changes.
-   Accesses appropriate documentation.
-   Generates the code or makes the edits.
-   Runs tools checking the syntax of the generated/edited program.
-   May run other tests.
-   Fixes the generated code when the tests fail.

----

### Thinking

Let's first understand what's meant by "thinking things through" by looking at [Claude Platform Docs (CPD)](https://platform.claude.com/docs/en/home) and their explanation of "Thinking" and then "Tools".

> **NOTE:** This area moves quickly. These classnotes were generated with the documentation as it read on November 6, 2026. See [Thinking:Overview](https://platform.claude.com/docs/en/build-with-claude/thinking), which is where you can find the following quotes in context.

-   No thinking: A model directly answers your prompt. As CPD states:
    - With "no scratch work, no checking, no changing approach halfway through".
    - Typically with poor results for complex or tricky tasks.
-   With thinking:
    - "[The model] works through the problem in its own words before answering."
    - In other words, "it restates what is being asked, tries approaches, checks intermediate results, and abandons paths that do not hold up."
    - "That up-front thinking arrives in `thinking` content blocks ..., and [the model] draws on it to produce the final answer."

> **NOTE:** Thinking consumes output tokens in your AI budget even when the thinking isn't displayed. It often isn't fully available to you. You can ask to see summaries of this thinking.

The following is an example of a model response after having done up-front thinking:

In [ ]:
{
  "content": [
    {
      "type": "thinking",
      "thinking": "Let me break this down. The question has two parts, so I'll start with the simpler one and use its result to constrain the second...",
      "signature": "WaUjzkypQ2mUEVM36O2Txu...."
    },
    {
      "type": "text",
      "text": "Based on my analysis..."
    }
  ]
}

----

### Turns

Thinking is typically improved with access to **tools** (e.g., internet searches and application-specific programs).

An **assistant turn** in Anthropic Claude models is defined as starting with a user prompt and ending "when Claude completes its full response, which might include multiple tool calls and results."

> **NOTE:** Assistant refers to the model. This definition of "turn" is _not_ the same as the one in Sid's talk.

See `m15/images/how_thinking_works.png` (from [CPD](https://platform.claude.com/docs/en/build-with-claude/thinking)), which illustrates the interaction of thinking and tools.

For these classnotes, we'll convert this kind of picture into what the user sees:

And see `m15/images/codegen_example.png` (from [Claude 3.7 Sonnet System Card](https://www.anthropic.com/claude-3-7-sonnet-system-card)), which illustrates the difference between an output with and without thinking enabled.

----
----

### From Single Tool Call to Agentic Loop

With this foundation on AI thinking, tools, and turns, let's see how to build from a single tool call to an agentic loop. We'll follow the [tutorial in CPD](https://platform.claude.com/docs/en/agents-and-tools/tool-use/build-a-tool-using-agent). After the tutorial, we'll return to Sid's code.

The tutorial:

-   Builds a calendar-management agent, not unlike Sid's inbox-to-Google-calendar workflow (slides 19-20).
-   Creates the agent in stages (called rings).

----

### Ring 1: Single tool, single turn

You can find the code at [this link](https://platform.claude.com/docs/en/agents-and-tools/tool-use/build-a-tool-using-agent#ring-1-single-tool-single-turn). `m15/ring1.py` is a copy of the code at that URL so that we can reference line numbers.

**Architecture**

-   Python program is the harness around Claude Opus 5.5, which has always-on adaptive thinking with customizable effort levels (defaults to `medium`). [Model description](https://platform.claude.com/docs/en/models/opus-5-5/overview).
-   Python program is a client that communicates with Anthropic's model server.
-   Python program prints some of the interesting events in a turn.

**Gist of the Python program**

1.  Imports `json` and `anthropic` libraries. Lines 3-5.
2.  Creates a client object for conversing with the Anthropic server. Line 8.
3.  Builds and sends a message to Claude Opus 5.5 that includes:
    - The user's prompt, i.e., "Schedule a 30-minute sync with ..." Lines 41-54.
    - A tool that the model can choose to call. If it chooses to call this tool, it must create a JSON block according to the given schema. This JSON defines the inputs for the tool. Lines 10-39.
4.  Looks at the LLM's response. This response indicates that the tool should be run, which is how to interpret a `stop_reason` that is "tool_use". Lines 56-58.

> **NOTE:** There should be a check at this point in the Python program for a response for which `stop_reason` isn't `tool_use`, but the tutorial program assumes things are going to work as expected. And given this, the Python program:

5.  Walks through the response blocks looking for one whose `block.type` is `"tool_use"`. Again, the assumption is that there's only one. Line 62.
6.  Prints what tool should be invoked and the input to give that tool. Lines 63-64.
7.  Fakes the tool's result. Lines 66-68.
8.  Builds and sends a new message to Claude Opus 5.5 that includes:
    - All the prior messages, since the API is stateless. Memory is in the harness! Lines 73-83.
    - And the tool result, which is marked as coming from the "user". The `tool_use_id` must match so that the model can match its request for a tool run with that run's result. Remember that the prior response can request multiple runs of the same tool with different inputs. Lines 84-93.
9.  Prints the model's final natural-language answer to the user's original prompt. This is the end of the turn because `stop_reason` is "end_turn". Lines 97-101.

**User sees** a single response to a prompt that took a single turn:

----

### Ring 2: The agentic loop

You can find the code at [this link](https://platform.claude.com/docs/en/agents-and-tools/tool-use/build-a-tool-using-agent#ring-2-the-agentic-loop). `m15/ring2.py` is a copy of the code at that URL.

**Architecture:** Everything in Ring 1, plus:

-   Adds a while-loop that keeps running tools and feeding results back to the model until `stop_reason` is no longer `"tool_use"`. Line 65.
-   Doesn't rebuild the `messages` array in the harness, but simply append new messages to it. Lines 47-53, 69-81.
-   Adds a `run_tool` stub that fakes the running of a local tool. The only known local tool is `create_calendar_event`. Lines 37-44.

**Gist of the Python program:** Focus on the while-loop at the end of the Python program:

1.  Same assumptions about the number of tool_use requests in a response (just one, please). Line 66.
2.  Append the new message containing the result of the latest "tool_use" run. Lines 70-81.
3.  Send the updated message list back to the model. Lines 83-89.
4.  When the while-loop exits, find the "text" block and print it. Lines 91-92.

**User sees** a single response to a prompt that took multiple turns:

**ASK:** What causes the model to make multiple "tool_use" requests?

----

### Ring 3: Multiple tools, parallel calls

You can find the code at [this link](https://platform.claude.com/docs/en/agents-and-tools/tool-use/build-a-tool-using-agent#ring-3-multiple-tools-parallel-calls). `m15/ring3.py` is a copy of the code at that URL.

**Architecture:** Everything in Ring 2, plus: 

-   A new tool `list_calendar_events` that allows the model to check the user's schedule before creating a new event. Its execution is faked (lines 55-58).
-   This new tool requires a new schema to describe its inputs. The tool returns a list of all events on a given date. Lines 34-44.

**Gist of the Python program:** Notice that the while-loop has been refactored to remove the assumption that there's only one block in the response with type "tool_use". Lines 77-92.

**User sees** a single response to a prompt that involved multiple tools:

**ASK:** What is a turn?

----

### Ring 4: Error handling

You can find the code at [this link](https://platform.claude.com/docs/en/agents-and-tools/tool-use/build-a-tool-using-agent#ring-4-error-handling). `m15/ring4.py` is a copy of the code at that URL.

**Architecture:** Everything in Ring 3, plus: 

-   Error-checking code in the `create_calendar_event` tool, which rejects an event that has too many attendees (just an example). Lines 50-51.
-   Wraps the code in the agentic loop that calls `run_tool` in a try-except statement. Lines 83-101.

**Gist of the Python program:** When a tool run fails, Anthropic models expect that the tool's error message is sent back with the flag `"is_error": True`. Lines 98-99. The model will use the error message to try to correct the generated tool input or end the turn asking the user for help.

**User sees** a single response to a prompt that involved multiple tools:

----

### Ring 5: The tool runner SDK abstraction

You can find the code at [this link](https://platform.claude.com/docs/en/agents-and-tools/tool-use/build-a-tool-using-agent#ring-5-the-tool-runner-sdk-abstraction). `m15/ring5.py` is a copy of the code at that URL.

**Architecture:** Everything in Ring 4, except that:

-   The agentic loop no longer appears in your harness code. Lines 45-55.
    - The Anthropic `tool_runner` makes one or more API call, checks `stop_reason` after each call, runs tools and appends results when necessary.
    - It does these things until a final "text" message is produced that ends the turn.
-   You simply wrap your tool functions in a Python decorator. Lines 11-30, 33-42.

> A **Python decorator** is how you express in your program that you'd like to wrap a function with extra behavior, which can take place before and/or after this function's behavior.
> 
> In the Ring 5 example, `tool_runner` is doing stuff before and after the running of `create_calendar_event`. In the definition of the function `beta_tool`, you'd see something like this:

----
----

### Choices in Tool Use

The tutorial used **client tools**, which were scripts run on the client's machine.

Claude also supports:

-   **Anthropic-defined schemas** for running common client tools like `bash` shell commands in the client's terminal and actions in the client's IDE (`text_editor`).
-   **Server tools** that run on Anthropic's infrastructure (for a cost). Examples include `web_search`, `web_fetch`, `code_execution`, and `tool_search`.

See [this CPD page](https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview) for more information.

As Sid's talk mentions, **parallelism** is an aspect of tool use.

-   Claude may include several `tool_use` blocks in a single response (i.e., assistant turn).
-   [CPD states](https://platform.claude.com/docs/en/agents-and-tools/tool-use/parallel-tool-use#execution-semantics), "How you run those calls is your decision. The API doesn’t prescribe an execution order: you can run the calls concurrently ..., sequentially in the order they appear, or in any combination that suits your tools."

> This is tricky ground. More soon.

----
----

### Sid's `inbox_calendar` Workflow

See slide 13 ("Most steps need no model at all"), which highlights the use of tools (called "Code" in the slide's image key).

So, let's dig into this `inbox_calendar` workflow, as illustrated on slide 20.

-   Workflow inputs: 18 messages and your calendar for the next 30 days.
-   Workflow outputs: your updated calendar and a couple of other files, which are necessary if you're going to approve the proposed changes to your calendar before they're applied.
-   Everything in the middle represents the code in `workflows/inbox_calendar.py`.
-   For now, we'll just focus on the AI dubbed "Master", which you can think of as analogous to Claude Opus 5.5 in the Anthropic `ring` programs.

----

### Building a Local Model

> Remember, when you install LM Studio, it will download open models to your machine and help your code interact with them.

In `inbox_calendar.py > main`, the script creates an `Agent` object called `master` from the Agent Card called "inbox_master". Lots of other unimportant code redacted.

In [ ]:
def main():  # in inbox_calendar.py
    
    tools = default_toolbox(OUT)   # where OUT is the output directory

    def agent(name):
        return Agent(AgentCard.load(name), tools)

    master = agent("inbox_master")

-   `default_toolbox` is defined at the bottom of `agentkit/tools.py`. It registers the function `resolve_date` and describes a schema for this function's input parameters. It looks like the schema you saw in `ring1.py`.
-   `Agent` constructs an object that stores an `AgentCard` and a list of tools (and other stuff). This class is defined in `agentkit/agent.py`.
-   You should start to understand that the folder `agentkit` contains code that builds agents, registers tools, and helps agents coordinate with each other.
-   Agent cards are described on slide 15 and stored in the folder `agents`. `inbox_master.json` states how the inbox-master agent should act, and its contents are shown below. You can start to see how the "instructions" lead to the arrows on Slide 20.

In [ ]:
{
  "name": "inbox_master",
  "tier": "master",
  "reasoning": "none",
  "temperature": 0.4,
  "job": "Plan the inbox run, settle escalated items, and write the student's daily digest, including what changed on the calendar.",
  "instructions": "You lead two teams: an Intake team (classifies every message, extracts dated items) and a Scheduling team (merges items and reconciles them with the student's calendar). You see summaries, not raw inboxes, except for messages escalated to you.\nWhen asked to PLAN: you have only seen subject lines, so plan at the level of policy. Choose the digest horizon in days and give each team one short instruction about HOW to work (e.g. what to watch for). Never name specific message ids, dates or categories you have not read; the teams read the messages, you don't.\nWhen asked to SETTLE an escalation: read the original message and return the corrected item. Use the resolve_date result you are given; do not do date arithmetic yourself.\nWhen asked to WRITE the digest: Markdown, at most 350 words: 'Today' (what must happen today), 'This week' (deadlines in time order with weekday and time), 'Events', 'Calendar' (one line per change: added, moved from old to new time, removed; say clearly if the changes are only proposed; then any overlaps and by-hand steps you are given), 'FYI' (one line each), 'Ignored' (count of promo/social). Be concrete. Never invent items or changes.",
  "success_check": "Digest lists every answer-key deadline in the horizon, every calendar change, and nothing extra.",
  "max_tokens": 12000
}

Step 1 on Slide 20 says, "Plans the run from subject lines only." This happens in `inbox_calendar.py` in the following code. `Agent.run` is the method that launches the model configured as `master`.

In [ ]:
# 0. The master plans from the headers only: a few hundred tokens, not the whole inbox.
headers = "\n".join(f"{m['id']} | {m['received']} | {m['from_name']} | {m['subject']}" for m in messages)
plan = master.run(f"PLAN. Current time: {NOW}. Goal: tomorrow-morning-ready digest and an up-to-date calendar "
                  f"for the student.\nInbox headers:\n{headers}", label="plan", schema=PLAN_SCHEMA)

Inside the method `Agent.run`, you'll find a tool loop, which allows the model to call tools until it has what it needs. This should remind you of the while-loop in the Anthropic `ring` programs. (Note: I redacted some of the uninteresting work.)

In [ ]:
# 1) Tool loop: let the model call tools until it stops asking for them.
reply = None
for _ in range(c.max_tool_turns if specs else 1):
    # ... prompt model and put response in reply ...

    for call in reply.tool_calls:
        result = self.toolbox.call(call.function.name, call.function.arguments,
                                    tracer=self.tracer, agent=c.name, label=label)
        messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

----

### Running a Workflow

To process your inbox and have the appointments appear in your calendar, you run the following under a Python virtual environment with `requirements.txt` installed:

In [ ]:
python3 workflows/inbox_calendar.py

----
----

### Commit Your Notes and Edits

Don't forget to review, commit, and push your edits so that you've updated your GitHub repo with today's work.

*   `git status` to review what's new and what's been edited.
*   `git add <list_of_files>` to indicate what you want committed.
*   `git commit -m "<your_commit_message>"` to commit the changes to the codespace's filesystem.
*   `git push` to push those committed changes to your GitHub repo.
*   `git status` to verify that you haven't forgotten anything.

Version 20261108